# Task 3 — Gender G-WD1 weight-decay screen

This experiment targets overfitting: a model doing very well on training images but worse on unseen images. **Run All trains only folds 0 and 4**, from scratch, then stops with a saved decision.

Use a Colab GPU (L4 was used for G2). Upload or publish this notebook and the changed source files to the selected repository branch before running the setup below. No local PyTorch install is needed.


## 1. Colab GPU and repository


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "fashion-analysis-and-cleanup"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"

def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repository ready:", REPO_DIR)
print("Commit:", commit)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git fetch origin task-3-gender-usage-classification
$ git switch task-3-gender-usage-classification
$ git merge --ff-only origin/task-3-gender-usage-classification
Repository ready: /content/MLA2
Commit: 0282673043dbdb770375c5260510282cdfcd0e63


## 2. Teacher data and canonical split


In [2]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()
    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            expected_bytes = DATA_ZIP.stat().st_size
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)
            if partial.stat().st_size != expected_bytes:
                raise OSError("The local ZIP copy is incomplete.")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except OSError as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError("Drive disconnected three times. Remount and retry.") from error
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)

copy_teacher_zip_to_local_disk()
teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        archive.extractall(REPO_DIR)

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
if actual_images != expected_images or not all(path.is_file() for path in required_files):
    raise RuntimeError(f"Teacher data is incomplete: {actual_images:,}/{expected_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")


Teacher data ready: 44,441 images


## 3. One change: stronger weight decay

Start a new GeM model from scratch using G2's recipe. Change only AdamW **weight decay from 0.0001 to 0.01**, on the same parameter groups. Weight decay discourages large fitted weights. This fixed value is a test choice, not a proven best value.

Keep G2's ±2-pixel translation, full images, original fold-training RGB statistics, GeM p=3, batch 128, FP32, 30 epochs, seed 2753, cross-entropy, learning-rate schedule and final-epoch checkpoint. **No G-D1 darkening, CPU offload, early stopping or new split.** G2 remains a comparison model; this does not reverse its earlier rejection.

GPU memory must stay **strictly below 3,000,000,000 bytes (3 GB)**. Time and latency are saved without speed caps. Every training run is registered. A completed fold that exceeds the memory limit stops the screen before another fold starts.

The source check below reads and verifies the five saved G2 and five matched E6 bundles without training. A missing or changed source stops the run. Results go to a new `experiments/t3_gender_weight_decay_001/gender` folder on Drive.


In [3]:
from fashion.train.task3_gender_weight_decay import (
    check_weight_decay_sources, run_gender_weight_decay_screen,
)

G2_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_v2_g2_translation/gender"
E6_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_e6_gem_p3/gender"
for path in (G2_DIR, E6_DIR, DRIVE_REGISTRY):
    if not path.exists():
        raise FileNotFoundError(f"Complete saved source evidence is required: {path}")

sources, gender_classes, candidate_spec = check_weight_decay_sources(
    g2_directory=G2_DIR, e6_directory=E6_DIR,
    source_registry_path=DRIVE_REGISTRY, root=REPO_DIR,
)
print("Verified sources:", {name: len(runs) for name, runs in sources.items()})
print("Weight decay:", candidate_spec.weight_decay)
print("Training augmentation:", candidate_spec.training_augmentation)
print("Output:", DRIVE_TASK_DIR / candidate_spec.artifact_dir / "gender")


Verified sources: {'G2': 5, 'E6': 5}
Weight decay: 0.01
Training augmentation: translation_uniform_2px_p05
Output: /content/drive/MyDrive/MLA2/task3/experiments/t3_gender_weight_decay_001/gender


## 4. Required result: better validation and a smaller gap

Measure the **same finished checkpoint** on unchanged training images and clean validation images, with dropout disabled and BatchNorm in evaluation mode. Do not substitute online augmented-training scores for this clean-training check.

All screen rules must pass:

- Pooled validation macro-F1 improves by at least **0.010 versus matched G2** (about 0.755726 here). Its paired family-bootstrap 95% lower bound must be **above zero**. Neither fold may lose more than 0.005 validation F1.
- The mean clean training–validation gap falls by at least **0.030 versus G2** (to about 0.218385 or below here). Neither fold's gap may grow.
- No pooled class loses more than 0.020 F1. NLL may worsen by at most 0.020 and ECE by at most 0.010 versus G2.
- Retain the original corruption guards versus matched E6: translation-induced loss improves by at least 0.030; each other corruption, including darkening, may worsen by at most 0.020. Each change is measured relative to that model's clean score.
- Keep 390,181 trainable parameters and GPU memory below 3 GB. Canonical IDs, families, configuration, registry and saved artifacts must verify.

Use 10,000 paired whole-family bootstrap draws within folds, seed 2753. Derive thresholds from the exact matched G2 results, not the rounded numbers above.

**A smaller gap caused only by worse training scores is not success.** Validation must improve too. A validation gain with no required gap reduction also fails. Always show training F1, validation F1 and the gap together.


In [4]:
gender_weight_decay = run_gender_weight_decay_screen(
    g2_directory=G2_DIR, e6_directory=E6_DIR,
    source_registry_path=DRIVE_REGISTRY,
    output_root=DRIVE_TASK_DIR, registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,), root=REPO_DIR, device_name="cuda",
)
print("Screen decision:", gender_weight_decay["status"])
if "reason" in gender_weight_decay:
    print(gender_weight_decay["reason"])
for row in gender_weight_decay.get("folds", []):
    print("Fold", row["fold"], "clean training F1:", row["candidate_train_f1"],
          "validation F1:", row["candidate_validation_f1"], "gap:", row["candidate_gap"])
for check in gender_weight_decay.get("checks", []):
    if check["status"] != "pass":
        print(check)


[task3] preparing target=gender fold=0: train=26,220 (before selection=26,220), validation=6,553
[task3] fitting fold-training RGB statistics for target=gender fold=0
[task3] RGB statistics ready for target=gender fold=0
[task3] registered t3_gender_weight_decay_001_gender_smallcnngem3_f0_s2753_7f11ebe1ad87_20260905T060619Z34fc37; the first optimiser step may now run
[task3] target=gender fold=0 epoch=1/30 train_loss=0.5835 train_macro_f1=0.4560 validation_loss=0.5191 validation_macro_f1=0.4016
[task3] target=gender fold=0 epoch=2/30 train_loss=0.4435 train_macro_f1=0.6019 validation_loss=0.4344 validation_macro_f1=0.6421
[task3] target=gender fold=0 epoch=3/30 train_loss=0.3899 train_macro_f1=0.6618 validation_loss=0.3811 validation_macro_f1=0.6834
[task3] target=gender fold=0 epoch=4/30 train_loss=0.3562 train_macro_f1=0.6881 validation_loss=0.3705 validation_macro_f1=0.6541
[task3] target=gender fold=0 epoch=5/30 train_loss=0.3295 train_macro_f1=0.7195 validation_loss=0.3904 validat

## 5. Stop and review

Do not start folds 1–3 automatically. A failed screen ends this candidate. A passing screen permits a separately planned confirmation, requiring positive clean paired intervals against both G2 and E6. Reused development folds are not independent test evidence. Keep the held-out test sealed.

Read `screen_decision.json`, `clean_gap_comparison.csv`, and pooled `oof_predictions.csv` in the new experiment folder. Each fold also saves its checkpoint, configuration, training history, final clean training/validation metrics, probabilities and corruption scores. Reusing a completed run requires another artifact and configuration check.
